<a href="https://colab.research.google.com/github/marsya505/DataMining/blob/main/Week12-DatasetSiswa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install graphviz -q
!pip install graphviz -q

In [ ]:
from google.colab import files
print("Upload file datakelas.csv:")
uploaded = files.upload()

import pandas as pd
df = pd.read_csv("datakelas.csv")
print(df)

In [ ]:
# Preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le_gender  = LabelEncoder()
le_vehicle = LabelEncoder()

df['Gender_enc']  = le_gender.fit_transform(df['Gender'])
df['Vehicle_enc'] = le_vehicle.fit_transform(df['Vehicle'])
df['GPA_cat']     = (df['GPA'] >= 3.7).astype(int)

features      = ['Age', 'Gender_enc', 'Height', 'Weight', 'Fuel/month', 'Vehicle_enc']
feature_names = ['Usia', 'Gender', 'Tinggi Badan', 'Berat Badan', 'Uang Bensin/bln', 'Kendaraan']
class_names   = ['GPA Rendah', 'GPA Tinggi']

X = df[features].values
y = df['GPA_cat'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Distribusi target -> Rendah:", sum(y==0), "| Tinggi:", sum(y==1))

In [ ]:
# Training Decision Tree + Export .dot
import os
from sklearn.tree import DecisionTreeClassifier, export_graphviz

IMAGES_PATH = os.path.join(".", "images")
os.makedirs(IMAGES_PATH, exist_ok=True)

tree_clf = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_clf.fit(X_train, y_train)

export_graphviz(
    tree_clf,
    out_file=os.path.join(IMAGES_PATH, "siswa_tree.dot"),
    feature_names=feature_names,
    class_names=class_names,
    rounded=True,
    filled=True
)

print("Decision Tree selesai ditraining.")

In [ ]:
# Visualisasi Decision Tree
from graphviz import Source

Source.from_file(os.path.join(IMAGES_PATH, "siswa_tree.dot"))

In [ ]:
# Akurasi Decision Tree
from sklearn.metrics import accuracy_score

y_pred_tree = tree_clf.predict(X_test)
print("Akurasi Decision Tree:", accuracy_score(y_test, y_pred_tree))

In [ ]:
# KNN Minkowski (berbagai nilai K)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

k_values = [1, 3, 5, 7, 9, 11]

print("KNN Minkowski (p=2)")
for k in k_values:
    classifier = KNeighborsClassifier(n_neighbors=k, metric='minkowski', p=2)
    acc = cross_val_score(classifier, X, y, cv=5, scoring='accuracy').mean()
    print(f"K={k} -> Akurasi: {acc:.2f}")

In [ ]:
# KNN Euclidean (berbagai nilai K)
for k in k_values:
    classifier = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    acc = cross_val_score(classifier, X, y, cv=5, scoring='accuracy').mean()
    print(f"K={k} -> Akurasi: {acc:.2f}")

In [ ]:
# KNN Best Model (K=9) + Prediksi
classifier = KNeighborsClassifier(n_neighbors=9, metric='minkowski', p=2)
classifier.fit(X_train, y_train)

predictions = classifier.predict(X_test)
print("Akurasi KNN (K=9, Minkowski):", accuracy_score(y_test, predictions))

In [ ]:
# Prediksi Manual
# Format: [Usia, Gender(0=F/1=M), TinggiBadan, BeratBadan, UangBensin, Kendaraan(Motor=1)]
contoh = [[20, 1, 170, 65, 150000, 1]]

hasil_tree = tree_clf.predict(contoh)
hasil_knn  = classifier.predict(contoh)

print("Input            :", contoh[0])
print("Decision Tree -> ", class_names[hasil_tree[0]])
print("KNN (K=9)     -> ", class_names[hasil_knn[0]])